# Simple CSV RAG - Groq (free, open-source Llama) + local embeddings
- LLM: Groq `openai/gpt-oss-120b` (auto-selected) (free API key: https://console.groq.com/keys)
- Embeddings: sentence-transformers `all-MiniLM-L6-v2` (local, free, no key)

In [ ]:
%pip install -r requirements.txt

## Imports and Groq key

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
key = os.getenv("GROQ_API_KEY", "")
if not key or "PASTE_YOUR" in key:
    key = getpass("Paste your Groq API key (gsk_...): ").strip()
os.environ["GROQ_API_KEY"] = key

## Download the CSV

In [ ]:
os.makedirs("data", exist_ok=True)
file_path = "data/customers-100.csv"
if not os.path.exists(file_path):
    url = "https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    Path(file_path).write_bytes(r.content)

data = pd.read_csv(file_path)
print(data.head())

## Load and split

In [ ]:
docs = CSVLoader(file_path=file_path).load_and_split()

## Embeddings + FAISS (local, free)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embeddings)

## LLM (Groq) - auto-picks a model that currently exists on your account

In [ ]:
from groq import Groq

available = {m.id for m in Groq().models.list().data}
preferred = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "llama-3.3-70b-versatile", "llama-3.1-8b-instant"]
model_name = next((m for m in preferred if m in available), None)
if model_name is None:
    raise RuntimeError(f"None of {preferred} available. Pick one from: {sorted(available)}")
print("Using Groq model:", model_name)
llm = ChatGroq(model=model_name, temperature=0)

## Retrieval chain

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])
rag_chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, prompt))

## Ask a question

In [ ]:
answer = rag_chain.invoke({"input": "Which company does Sheryl Baxter work for?"})
print(answer["answer"])